# Phase 6 — Improvement Strategies: Adaptive & Progressive Resolution

Two strategies to close the gap between cheap (224 px) and expensive (518 px) inference:

| Strategy | Idea |
|----------|------|
| **Adaptive Resolution** | Global low-res pass -> uncertain patches -> high-res patch inference -> merge |
| **Progressive Refinement** | Coarse-to-fine pyramid (224->336->518 px) blending depth maps |

**Baseline comparison:** pure 224 px, pure 518 px, adaptive, progressive.

## 0 — Setup

In [ ]:
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def git(*args):
    subprocess.check_call(["git"] + list(args))

pip("--upgrade", "numpy")
pip("plotly", "pandas", "pillow", "scipy", "matplotlib")

if not os.path.isdir("vggt"):
    git("clone", "--depth", "1", "https://github.com/facebookresearch/vggt.git")
    pip("-e", "vggt")

if not os.path.isdir("vggt-eval"):
    git("clone", "--depth", "1", "https://github.com/insha-py/vggt-eval.git")
else:
    git("-C", "vggt-eval", "pull", "--ff-only")

sys.path.insert(0, "vggt-eval")
sys.path.insert(0, "vggt")

print("Packages installed. Restarting kernel ...")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Restart the kernel manually, then run from the Imports cell.")

In [ ]:
import os, sys, time
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.tum_vi                   import TUMVIDataset
from src.pipeline                 import VGGTPipeline, load_images_from_dir, run_vggt_inference
from src.improvements.adaptive    import AdaptiveResolutionVGGT
from src.improvements.progressive import ProgressiveRefinement
from src.metrics                  import (
    compute_ate, compute_rpe, compute_rotation_errors,
    MemoryProfiler, Timer, print_metrics_table,
)

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1 — Download TUM-VI `room1`

In [ ]:
SEQUENCE     = "room1"
N_FRAMES     = 40
DOWNLOAD_DIR = "/tmp/tumvi"

ds   = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()
data = ds.load()

image_dir     = data["image_dir"]
gt_extrinsics = data["gt_extrinsics"]   # (N, 3, 4)
N = len(gt_extrinsics)
print(f"Loaded {N} frames")

## 2 — Baselines: Pure 224 px and Pure 518 px

In [ ]:
pipe = VGGTPipeline()
pipe.load_model()

def run_at_res(res):
    imgs, _, _ = load_images_from_dir(image_dir, target_size=res, max_frames=N_FRAMES)
    with MemoryProfiler() as mem, Timer() as tmr:
        out = run_vggt_inference(pipe.model, imgs, pipe.device, pipe.dtype, resolution=res)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out, tmr.elapsed, mem.peak_mb

print("Running 224 px baseline ...")
out_224, t_224, mb_224 = run_at_res(224)

print("Running 518 px baseline ...")
out_518, t_518, mb_518 = run_at_res(518)

print(f"\n224 px: {t_224:.1f}s   {mb_224:.0f} MB")
print(f"518 px: {t_518:.1f}s   {mb_518:.0f} MB")

## 3 — Adaptive Resolution

In [ ]:
adaptive = AdaptiveResolutionVGGT(
    global_res     = 224,
    patch_res      = 518,
    conf_threshold = 0.3,
    patch_size_px  = 64,
    max_patches    = 4,
)
# Reuse the already-loaded model
adaptive._model  = pipe.model
adaptive._device = pipe.device
adaptive._dtype  = pipe.dtype

print("Running Adaptive Resolution ...")
t0 = time.perf_counter()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

out_adapt = adaptive.run(image_dir=image_dir, max_frames=N_FRAMES)

t_adapt  = time.perf_counter() - t0
mb_adapt = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Adaptive: {t_adapt:.1f}s   {mb_adapt:.0f} MB   patches_used={out_adapt['patches_used']}")

## 4 — Progressive Refinement

In [ ]:
progressive = ProgressiveRefinement(
    pyramid    = [224, 336, 518],
    blend_mode = "conf_weighted",
)
progressive._model  = pipe.model
progressive._device = pipe.device
progressive._dtype  = pipe.dtype

print("Running Progressive Refinement (224 -> 336 -> 518) ...")
t0 = time.perf_counter()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

out_prog = progressive.run(image_dir=image_dir, max_frames=N_FRAMES)

t_prog  = time.perf_counter() - t0
mb_prog = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Progressive: {t_prog:.1f}s   {mb_prog:.0f} MB")

## 5 — Pose Metrics Comparison

In [ ]:
def metrics_row(ext, label, time_s, peak_mb):
    gt = gt_extrinsics[:len(ext)]
    ate = compute_ate(ext, gt, align=True, with_scale=True)
    rpe = compute_rpe(ext, gt, step=1)
    rot = compute_rotation_errors(ext, gt)
    return dict(
        Method       = label,
        ATE_mean     = round(ate["mean"],        4),
        ATE_rmse     = round(ate["rmse"],        4),
        ATE_median   = round(ate["median"],      4),
        RPE_t_mean   = round(rpe["trans_mean"],  4),
        RPE_r_mean   = round(rpe["rot_mean"],    2),
        Rot_mean_deg = round(rot["mean"],        2),
        time_s       = round(time_s,  1),
        peak_mb      = round(peak_mb, 0),
    )

rows = [
    metrics_row(out_224["extrinsic"],   "224 px (low baseline)",      t_224,   mb_224),
    metrics_row(out_518["extrinsic"],   "518 px (high baseline)",     t_518,   mb_518),
    metrics_row(out_adapt["extrinsic"], "Adaptive (224+518 patches)", t_adapt, mb_adapt),
    metrics_row(out_prog["extrinsic"],  "Progressive (224->336->518)",t_prog,  mb_prog),
]

df = pd.DataFrame(rows)
print(f"\n=== Results: TUM-VI {SEQUENCE} ({N} frames) ===")
print(df.to_string(index=False))

os.makedirs("results", exist_ok=True)
df.to_csv(f"results/phase6_improvements_{SEQUENCE}.csv", index=False)
print("\nSaved to results/phase6_improvements_room1.csv")

## 6 — ATE Bar Chart

In [ ]:
colors = ["royalblue", "firebrick", "darkorange", "forestgreen"]
fig = go.Figure()
for i, row in df.iterrows():
    fig.add_trace(go.Bar(
        x=[row["Method"]], y=[row["ATE_mean"]],
        name=row["Method"],
        marker_color=colors[i],
        error_y=dict(type="data",
                     array=[row["ATE_rmse"] - row["ATE_mean"]],
                     visible=True),
    ))
fig.update_layout(
    title=f"ATE Comparison -- TUM-VI {SEQUENCE}",
    yaxis_title="ATE mean (m)",
    showlegend=False,
    bargap=0.3,
)
fig.show()

## 7 — Depth Map Visual Comparison

In [ ]:
import torch.nn.functional as F
import torch as th

FRAME_IDX = 0

depth_low   = out_224["depth_map"][FRAME_IDX, :, :, 0]
depth_high  = out_518["depth_map"][FRAME_IDX, :, :, 0]
depth_adapt = out_adapt["depth_map"][FRAME_IDX, :, :, 0]
depth_prog  = out_prog["depth_map"][FRAME_IDX, :, :, 0]

conf_low  = out_224["depth_conf"][FRAME_IDX]
conf_high = out_518["depth_conf"][FRAME_IDX]
conf_low_orig = out_adapt.get("conf_map_low", out_224["depth_conf"])[FRAME_IDX]
conf_prog = out_prog["depth_conf"][FRAME_IDX]

def to_display(arr, size=224):
    t = th.from_numpy(arr.astype("float32")).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t.squeeze().numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
pairs = [
    (depth_low,   conf_low,       "Depth 224 px",    "Conf 224 px"),
    (depth_high,  conf_high,      "Depth 518 px",    "Conf 518 px"),
    (depth_adapt, conf_low_orig,  "Depth Adaptive",  "Conf 224 (pre-patch)"),
    (depth_prog,  conf_prog,      "Depth Progressive","Conf Progressive"),
]
for col, (d, c, dt, ct) in enumerate(pairs):
    axes[0, col].imshow(to_display(d), cmap="plasma")
    axes[0, col].set_title(dt, fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(to_display(c), cmap="viridis", vmin=0, vmax=1)
    axes[1, col].set_title(ct, fontsize=9)
    axes[1, col].axis("off")

plt.suptitle(f"Depth & Confidence -- Frame {FRAME_IDX}", y=1.01)
plt.tight_layout()
plt.show()

## 8 — Depth Confidence Statistics

In [ ]:
methods_conf = [
    ("224 px",      out_224["depth_conf"]),
    ("518 px",      out_518["depth_conf"]),
    ("Adaptive",    out_adapt["depth_conf"]),
    ("Progressive", out_prog["depth_conf"]),
]

print(f"\n{'Method':<18}  {'Mean conf':>10}  {'Median conf':>12}  {'<0.3 frac':>10}")
print("-" * 58)
for name, conf in methods_conf:
    print(f"{name:<18}  {conf.mean():>10.4f}  {float(np.median(conf)):>12.4f}  "
          f"{float((conf < 0.3).mean()):>10.4f}")

## 9 — Time vs Accuracy Trade-off

In [ ]:
fig = px.scatter(
    df, x="time_s", y="ATE_mean",
    text="Method", color="Method",
    title="Accuracy-Efficiency Trade-off (lower-left = better)",
    labels={"time_s": "Inference time (s)", "ATE_mean": "ATE mean (m)"},
    size=[12] * len(df),
)
fig.update_traces(textposition="top center")
fig.show()

## 10 — Trajectories: All Methods vs GT

In [ ]:
from src.metrics import align_trajectories_umeyama

def cam_centers(ext):
    R = ext[:, :3, :3]
    t = ext[:, :3,  3]
    return -np.einsum("nij,nj->ni", R.transpose(0, 2, 1), t)

c_gt   = gt_extrinsics[:, :3, 3]
c_224  = cam_centers(out_224["extrinsic"])
c_518  = cam_centers(out_518["extrinsic"])
c_adap = cam_centers(out_adapt["extrinsic"])
c_prog = cam_centers(out_prog["extrinsic"])

R_al, t_al, s_al = align_trajectories_umeyama(c_518, c_gt, with_scale=True)

def align(c):
    return s_al * (R_al @ c.T).T + t_al

def traj(c, name, color, dash="solid"):
    return go.Scatter3d(
        x=c[:, 0], y=c[:, 1], z=c[:, 2],
        mode="lines+markers",
        name=name,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=3),
    )

fig = go.Figure([
    traj(c_gt,          "Ground truth",       "black"),
    traj(align(c_224),  "224 px",             "steelblue",   "dash"),
    traj(align(c_518),  "518 px",             "firebrick"),
    traj(align(c_adap), "Adaptive",           "darkorange",  "dot"),
    traj(align(c_prog), "Progressive",        "forestgreen", "longdash"),
])
fig.update_layout(
    title=f"Trajectories -- TUM-VI {SEQUENCE}",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
)
fig.show()

## Discussion

### Expected trade-off

| Method       | ATE     | Time       | Memory  |
|--------------|---------|------------|---------|
| 224 px       | worst   | fastest    | lowest  |
| Adaptive     | ~518 px | medium     | low-med |
| Progressive  | ~518 px | ~3x 224 px | medium  |
| 518 px       | best    | slowest    | highest |

### Limitations

- Adaptive patching refines depth only; camera poses still come from the global 224-px pass.
- Progressive refinement triples inference cost — worthwhile only when 518-px baseline saturates VRAM.
- A learned uncertainty estimator would outperform the naive low-confidence patch heuristic.